# Netflix Movies & TV Shows Analysis
This notebook contains data cleaning, analysis, and interactive visualizations for Netflix titles dataset.

In [ ]:
# ================================
# Netflix Movies & TV Shows Analysis
# ================================

# Import Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import matplotlib.pyplot as plt

# Set Plotly Renderer
pio.renderers.default = "browser"

# ================================
# Load Dataset
# ================================

df = pd.read_csv("netflix_titles.csv")

# Display first 5 rows
print(df.head())

# ================================
# Data Cleaning
# ================================

# Check missing values
print("\nMissing Values:\n")
print(df.isnull().sum())

# Clean date column
df['date_added'] = df['date_added'].astype(str).str.strip()

# Convert to datetime safely
df['date_added'] = pd.to_datetime(
    df['date_added'],
    errors='coerce'
)

# Fill missing values
df['director'] = df['director'].fillna("Unknown")
df['cast'] = df['cast'].fillna("Unknown")
df['country'] = df['country'].fillna("Unknown")

# Fill missing ratings with mode
df['rating'] = df['rating'].fillna(
    df['rating'].mode()[0]
)

# Drop rows with missing date or duration
df.dropna(
    subset=['date_added', 'duration'],
    inplace=True
)

# ================================
# Fix Incorrect Rating Entries
# ================================

# Some duration values are mistakenly stored in rating column
mask = df['rating'].astype(str).str.contains(
    'min',
    na=False
)

# Move incorrect values to duration column
df.loc[mask, 'duration'] = df.loc[mask, 'rating']
df.loc[mask, 'rating'] = np.nan

# Fill missing ratings again
df['rating'] = df['rating'].fillna(
    df['rating'].mode()[0]
)

# ================================
# Dataset Information
# ================================

print("\nDataset Info:\n")
print(df.info())

print("\nUnique Ratings:\n")
print(df['rating'].unique())

# ================================
# Data Analysis
# ================================

# Count content types
print("\nContent Types:\n")
print(df['type'].value_counts())

# Count ratings
print("\nRatings Count:\n")
print(df['rating'].value_counts())

# ================================
# Visualization 1:
# Movies vs TV Shows (Donut Chart)
# ================================

type_counts = df['type'].value_counts()

fig1 = px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='Share of Movies and TV Shows on Netflix',

    color_discrete_sequence=[
        "#00C6FF",
        "#7B2FF7"
    ],

    hole=0.45
)

fig1.update_traces(
    textposition='inside',
    textinfo='percent+label',

    insidetextfont=dict(
        color='white',
        size=15
    ),

    pull=[0.03, 0.03],

    marker=dict(
        line=dict(
            color='white',
            width=3
        )
    )
)

fig1.update_layout(
    title_font=dict(size=24),

    legend=dict(
        title='Content Type',
        font=dict(size=13)
    )
)

fig1.show()

# ================================
# Visualization 2:
# Netflix Content by Audience Rating
# ================================

rating_counts = (
    df['rating']
    .value_counts()
    .reset_index()
)

rating_counts.columns = ['Rating', 'Count']

fig2 = px.bar(
    rating_counts,
    x='Rating',
    y='Count',
    title='Netflix Content by Audience Rating',

    text_auto=True,

    color='Count',

    color_continuous_scale=[
        "#00C6FF",
        "#0072FF",
        "#7B2FF7"
    ]
)

fig2.update_layout(
    title_font=dict(size=24),

    xaxis_title='Audience Rating',
    yaxis_title='Number of Titles',

    font=dict(size=12)
)

fig2.show()

# ================================
# Visualization 3:
# Annual Content Releases on Netflix
# ================================

df['year_added'] = df['date_added'].dt.year

year_counts = (
    df.groupby(['year_added', 'type'])
    .size()
    .reset_index(name='count')
)

fig3 = px.line(
    year_counts,
    x='year_added',
    y='count',
    color='type',

    title='Annual Content Releases on Netflix',

    markers=True,

    labels={
        'year_added': 'Year',
        'count': 'Number of Titles'
    },

    color_discrete_sequence=[
        "#00C6FF",
        "#7B2FF7"
    ]
)

fig3.update_traces(
    line=dict(width=4),
    marker=dict(size=9)
)

fig3.update_layout(
    title_font=dict(size=24),

    xaxis_title='Year',
    yaxis_title='Total Content Added',

    legend_title='Content Type'
)

fig3.show()

plt.savefig("netflix_analysis.pdf", dpi=300, bbox_inches='tight')

print(
    "\nThank you for exploring this Netflix Data Analysis Project!"
)
